# Qwen/Qwen3-VL-2B-Instruct: Multi-Prompting Strategies Evaluation on Colab GPU
### Strategies: Zero-Shot Direct vs. One-Shot vs. Zero-Shot Chain-of-Thought (CoT)
**Requirements**: Make sure your Colab Runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).


In [ ]:
# 1. Install required packages
!pip install -q -U git+https://github.com/huggingface/transformers.git accelerate qwen-vl-utils


In [ ]:
import os
import sys
import json
import time
import gzip
import base64
import re
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float32

model_id = "Qwen/Qwen3-VL-2B-Instruct"
print(f"Loading {model_id} on {device}...")
processor = AutoProcessor.from_pretrained(model_id)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None
)
model.eval()
print("Model loaded successfully!")


In [ ]:
# 3. Load Embedded Benchmark Data (100 Updated Items)
BENCHMARK_B64 = """..."""
items = json.loads(gzip.decompress(base64.b64decode(BENCHMARK_B64.encode("ascii"))).decode("utf-8"))
print(f"Loaded {len(items)} benchmark items.")


In [ ]:
# 4. Prompt Generation & Extraction Functions
def format_sg_components(sg):
    objects = sg.get("objects", [])
    relations = sg.get("relations", [])
    obj_lines = []
    for idx, obj in enumerate(objects, 1):
        lbl = obj.get("label", "unknown")
        attrs = obj.get("attributes", [])
        bbox = obj.get("bounding_box", [])
        attr_str = f"attributes: [{', '.join(attrs)}]" if attrs else "no attributes"
        bbox_str = f"bbox: [{bbox[0]:.2f}, {bbox[1]:.2f}, {bbox[2]:.2f}, {bbox[3]:.2f}]" if len(bbox) == 4 else ""
        details = ", ".join(filter(None, [attr_str, bbox_str]))
        obj_lines.append(f"  - {lbl} ({details})")

    rel_lines = []
    for rel in relations:
        s = rel.get("subject", "")
        r = rel.get("relation", "")
        o = rel.get("object", "")
        rel_lines.append(f'  - "{s}" {r} "{o}"')
        
    return len(objects), "\n".join(obj_lines) if obj_lines else "  None", len(relations), "\n".join(rel_lines) if rel_lines else "  None"

def build_prompt_zero_shot_direct(item):
    num_obj, obj_str, num_rel, rel_str = format_sg_components(item)
    q = item["query"].get("question", "")
    return (
        "You are given a scene graph representation of an image containing objects and their relationships, followed by a question.\n\n"
        f"Scene Graph:\nObjects ({num_obj}):\n{obj_str}\n\n"
        f"Relationships ({num_rel}):\n{rel_str}\n\n"
        f"Question: {q}\n\n"
        "Instructions: Based strictly on the scene graph provided above, answer the question accurately. "
        "Provide only the direct concise answer (a few words or a single term)."
    )

def build_prompt_one_shot(item):
    num_obj, obj_str, num_rel, rel_str = format_sg_components(item)
    q = item["query"].get("question", "")
    return (
        "You are given a scene graph representation of an image containing objects and their relationships, followed by a question. Answer with the direct concise target entity.\n\n"
        "Example:\n"
        "Scene Graph:\n"
        "Objects (4):\n"
        "  - table (attributes: [wooden, brown], bbox: [0.10, 0.40, 0.80, 0.50])\n"
        "  - laptop (attributes: [silver], bbox: [0.25, 0.35, 0.30, 0.25])\n"
        "  - cup (attributes: [white, ceramic], bbox: [0.65, 0.45, 0.12, 0.15])\n"
        "  - notebook (attributes: [blue], bbox: [0.42, 0.48, 0.18, 0.20])\n"
        "Relationships (3):\n"
        '  - "laptop" on "table"\n'
        '  - "cup" to the right of "laptop"\n'
        '  - "notebook" to the right of "laptop"\n'
        "Question: What is on the table to the right of the laptop and to the left of the cup?\n"
        "Answer: notebook\n\n"
        "Now answer the question for the following scene graph:\n"
        f"Scene Graph:\nObjects ({num_obj}):\n{obj_str}\n\n"
        f"Relationships ({num_rel}):\n{rel_str}\n\n"
        f"Question: {q}\n"
        "Answer:"
    )

def build_prompt_zero_shot_cot(item):
    num_obj, obj_str, num_rel, rel_str = format_sg_components(item)
    q = item["query"].get("question", "")
    return (
        "You are given a scene graph representation of an image containing objects and their relationships, followed by a question.\n\n"
        f"Scene Graph:\nObjects ({num_obj}):\n{obj_str}\n\n"
        f"Relationships ({num_rel}):\n{rel_str}\n\n"
        f"Question: {q}\n\n"
        "Instructions:\n"
        "1. Reason step-by-step:\n"
        "   - Identify the objects and attributes mentioned in the question.\n"
        "   - Trace the specified relationship(s) or spatial arrangement. If the question asks what is between two objects, check which candidate object's horizontal coordinate center lies between them.\n"
        "2. Provide your final concise answer on the last line in the exact format:\n"
        "   Final Answer: <answer>"
    )

def extract_answer(pred_text, strategy_name):
    clean = pred_text.strip()
    if strategy_name == "zero_shot_cot":
        match = re.search(r"Final Answer:\s*(.+)", clean, re.IGNORECASE)
        if match:
            ans = match.group(1).strip()
            return ans.split("\n")[0].strip()
        lines = [line.strip() for line in clean.split("\n") if line.strip()]
        if lines:
            last = lines[-1]
            last = re.sub(r"^(?:answer|the answer is|final answer):\s*", "", last, flags=re.IGNORECASE)
            return last.strip()
    elif strategy_name == "one_shot":
        first_line = clean.split("\n")[0].strip()
        first_line = re.sub(r"^(?:answer:)\s*", "", first_line, flags=re.IGNORECASE)
        return first_line.strip()
    return clean


In [ ]:
# 5. Run Evaluation Loop
def evaluate_strategy(strategy_name, prompt_fn, max_tokens):
    print(f"\n{'='*50}\nRunning: {strategy_name} (max_tokens={max_tokens})\n{'='*50}")
    start = time.time()
    results = []
    correct, exact = 0, 0
    t_stats = {}

    for i, item in enumerate(items, 1):
        q_info = item["query"]
        gt = str(q_info.get("answer", "")).strip().lower()
        t = q_info.get("template", "unknown")

        prompt = prompt_fn(item)
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]

        inputs = processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
        )
        if device == "cuda":
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        
        gen_ids = [o[len(inp):] for inp, o in zip(inputs["input_ids"], out_ids)]
        raw_pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

        parsed = extract_answer(raw_pred, strategy_name)
        clean_pred = parsed.lower().rstrip(".").strip()

        is_exact = (clean_pred == gt)
        is_correct = is_exact or (gt in clean_pred) or (clean_pred in gt and len(clean_pred) > 1)

        if is_correct: correct += 1
        if is_exact: exact += 1

        if t not in t_stats: t_stats[t] = {"total": 0, "correct": 0}
        t_stats[t]["total"] += 1
        if is_correct: t_stats[t]["correct"] += 1

        results.append({
            "image_number": item["image_number"],
            "gqa_image_id": item["gqa_image_id"],
            "template": t,
            "question": q_info.get("question"),
            "ground_truth": gt,
            "raw_output": raw_pred,
            "parsed_answer": parsed,
            "is_correct": is_correct
        })

        if i % 25 == 0 or i == len(items):
            print(f"[{i:03d}/100] Accuracy: {correct}/{i} ({correct/i*100:.1f}%) | Elapsed: {time.time()-start:.1f}s")

    elapsed = time.time() - start
    summary = {
        "strategy": strategy_name,
        "total": len(items),
        "correct": correct,
        "exact": exact,
        "accuracy": round(correct / len(items), 4),
        "time_seconds": round(elapsed, 2),
        "template_breakdown": {
            k: {"total": v["total"], "correct": v["correct"], "accuracy": round(v["correct"]/v["total"], 4)}
            for k, v in t_stats.items()
        }
    }
    
    with open(f"results_{strategy_name}.json", "w") as f:
        json.dump({"metadata": summary, "results": results}, f, indent=2)
        
    return summary

strategies = [
    ("zero_shot_direct", build_prompt_zero_shot_direct, 24),
    ("one_shot", build_prompt_one_shot, 24),
    ("zero_shot_cot", build_prompt_zero_shot_cot, 128)
]

summaries = {}
for name, pfn, tok in strategies:
    summaries[name] = evaluate_strategy(name, pfn, tok)

with open("strategies_summary.json", "w") as f:
    json.dump(summaries, f, indent=2)

print("\n=== Summary Comparison ===")
import pandas as pd
df = pd.DataFrame([
    {
        "Strategy": s["strategy"],
        "Overall Accuracy": f"{s['accuracy']*100:.1f}%",
        "attr_relation": f"{s['template_breakdown']['attr_relation']['accuracy']*100:.1f}%",
        "two_hop": f"{s['template_breakdown']['two_hop']['accuracy']*100:.1f}%",
        "between_attr": f"{s['template_breakdown']['between_attr']['accuracy']*100:.1f}%",
        "Time (s)": s["time_seconds"]
    }
    for s in summaries.values()
])
print(df.to_string(index=False))
